In [1]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plot
from sklearn.linear_model import LinearRegression
import sklearn.metrics
from sklearn.metrics import explained_variance_score, mean_squared_error, r2_score # https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html
import statsmodels.api as sm
import pandas
import numpy as np
import matplotlib.pyplot as plt
import pandas

# For 3d plots. This import is necessary to have 3D plotting below
from mpl_toolkits.mplot3d import Axes3D

# For statistics. Requires statsmodels 5.0 or more
from statsmodels.formula.api import ols
# Analysis of Variance (ANOVA) on linear models
from statsmodels.stats.anova import anova_lm

# Adapted from:
# https://www.datasciencecentral.com/profiles/blogs/linear-regression-in-python-use-of-numpy-scipy-and-statsmodels,
# https://realpython.com/linear-regression-in-python/#multiple-linear-regression


In [2]:
DataToFit = np.transpose(np.asarray(np.genfromtxt('EqualdeltalogtForHenry.csv',delimiter=',',skip_header=1)))
# 6 = logM
logMIndex = 6
# 7 = logL
logLIndex = 7
# 9 = logR
logRIndex = 9
# 14 = Fundamental period from Trabucchi model
P0Index = 13
# 15 = logP0 from Trabucchi model
logP0Index = 12
# 15 = Overtone period from Trabucchi model
P1Index=15
# 15 = logP1 from COLIBRI model
logP1Index=16
# 17 = Pulsation Mode
ModeIndex=17
# 25 = log(C/O)
logCORatioIndex=25

In [3]:
DataToFit[6]

array([-0.14509 , -0.087247, -0.12321 , ...,  0.39777 ,  0.39707 ,
        0.25406 ])

In [4]:
def PulsationModeSort (input_array):
    # DON'T TRUST THIS SORT METHOD. MODES IN DATA SET ARE UNRELIABLE
    FundamentalPulsators = []
    OvertonePulsators = []
    i = 0
    while i < len(input_array[ModeIndex]):
        if input_array[ModeIndex][i] == 0:
            FundamentalPulsators.append(i)
        if input_array[ModeIndex][i] == 1:
            OvertonePulsators.append(i)
        i += 1
    FundamentalModeStars = np.transpose(input_array.transpose()[FundamentalPulsators])
    FirstOvertoneModeStars = np.transpose(input_array.transpose()[OvertonePulsators])
    return FundamentalModeStars, FirstOvertoneModeStars

def CompositionSort (input_array):
    ORichStarsIndices = []
    CRichStarsIndices = []
    i = 0
    while i < len(input_array[logCORatioIndex]):
        if input_array[logCORatioIndex][i] < 0:
            ORichStarsIndices.append(i)
        if input_array[logCORatioIndex][i] > 0:
            CRichStarsIndices.append(i)
        i += 1
    ORichStars = np.transpose(input_array.transpose()[ORichStarsIndices])
    CRichStars = np.transpose(input_array.transpose()[CRichStarsIndices])
    return ORichStars, CRichStars

In [5]:
def MassSort (input_array):
    logMLT005 = []
    logMGT005 = []
    i = 0
    while i < len(input_array[logMIndex]):
        if input_array[logMIndex][i] <= 0.05: # For all stars with logM < 0.05
            logMLT005.append(i)
        if input_array[logMIndex][i] > -0.05 and input_array[6][i] <= 0.4 : # For all stars with logM > -0.05 - yes, there needs to be overlap!
            logMGT005.append(i)
        #if input_array[logMIndex][i] > 0.4:
            #print("Star", i, "of mass",10**(input_array[6][i]),"cut")
        i += 1
    #LowMassStars = np.transpose(input_array.transpose()[logMLT005])
    #HighMassStars = np.transpose(input_array.transpose()[logMGT005])
    return logMLT005, logMGT005

def MassFilter (Mass_Data):
    logMListArray = []
    i = 0
    while i<len(Mass_Data):
        if Mass_Data[i] < 0.4:
            logMListArray.append(i)
        i += 1
    return logMListArray

In [6]:
P0Stars, P1Stars = PulsationModeSort(DataToFit)
LowLogMToFitStars, HighLogMToFitStars = MassSort(DataToFit)
#ORichP0Stars, CRichP0Stars = CompositionSort(P0Stars)
#ORichP1Stars, CRichP1Stars = CompositionSort(P1Stars)
logMData = DataToFit[logMIndex]
logMLT005Data = DataToFit[logMIndex][LowLogMToFitStars]
logMMGT005Data = DataToFit[logMIndex][HighLogMToFitStars]
logRData = DataToFit[logRIndex]
logRMLT005Data = DataToFit[logRIndex][LowLogMToFitStars]
logRMGT005Data = DataToFit[logRIndex][HighLogMToFitStars]
logLData = DataToFit[logLIndex]
logLLT005Data = DataToFit[logLIndex][LowLogMToFitStars]
logLMGT005Data = DataToFit[logLIndex][HighLogMToFitStars]

In [8]:
# First, let's look at all of the Fundamental mode data at once.
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = DataToFit[logMIndex]
logRData = DataToFit[logRIndex]
logPData = DataToFit[logP0Index]

data = pandas.DataFrame({'logM': logMData, 'logR': logRData, 'logP': logPData})

# Fit the model
model = ols("logP ~ logM + logR", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                   logP   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                 2.811e+05
Date:                Tue, 11 Feb 2020   Prob (F-statistic):               0.00
Time:                        09:16:19   Log-Likelihood:                 3714.4
No. Observations:                1423   AIC:                            -7423.
Df Residuals:                    1420   BIC:                            -7407.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.2529      0.006   -362.443      0.0

M: logP0 = -2.253(6) - 0.776(3)logM + 2.030(3)logR

In [10]:
# Now, let's look at all of the 1st overtone mode data at once. This is a check to see if we can get the published Trabuccchi formula back
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = DataToFit[logMIndex]
logRData = DataToFit[logRIndex]
logPData = DataToFit[logP1Index]

data = pandas.DataFrame({'logM': logMData, 'logR': logRData, 'logP': logPData})

# Fit the model
model = ols("logP ~ logM + logR", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                   logP   R-squared:                       0.926
Model:                            OLS   Adj. R-squared:                  0.926
Method:                 Least Squares   F-statistic:                     8868.
Date:                Tue, 11 Feb 2020   Prob (F-statistic):               0.00
Time:                        09:21:37   Log-Likelihood:                 1833.2
No. Observations:                1423   AIC:                            -3660.
Df Residuals:                    1420   BIC:                            -3645.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.0737      0.023    -46.052      0.0

In [9]:
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logRData = logRMGT005Data
logLData = logLMGT005Data
logMData = logMMGT005Data

data = pandas.DataFrame({'x0': logLData, 'x1': logMData, 'y': logRData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.938
Model:                            OLS   Adj. R-squared:                  0.938
Method:                 Least Squares   F-statistic:                     9907.
Date:                Thu, 06 Feb 2020   Prob (F-statistic):               0.00
Time:                        09:42:26   Log-Likelihood:                 2241.3
No. Observations:                1318   AIC:                            -4477.
Df Residuals:                    1315   BIC:                            -4461.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.3994      0.020    -20.198      0.0

M Stars, logM>0: logR = -0.399(20) + 0.761(6)logL - 0.552(13)logM

In [10]:
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logRData = logRMLT005Data
logLData = logLLT005Data
logMData = logMLT005Data
logMDataSq = logMLT005Data**2

data = pandas.DataFrame({'x0': logLData, 'x1': logMData, 'x2':logMDataSq, 'y': logRData})

# Fit the model
model = ols("y ~ x0 + x1 + x2", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.866
Model:                            OLS   Adj. R-squared:                  0.864
Method:                 Least Squares   F-statistic:                     492.4
Date:                Thu, 06 Feb 2020   Prob (F-statistic):           1.53e-99
Time:                        09:42:26   Log-Likelihood:                 288.15
No. Observations:                 233   AIC:                            -568.3
Df Residuals:                     229   BIC:                            -554.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.8187      0.082    -10.038      0.0

M Stars, logM<0: logR = -0.819(82) + 0.869(23)logL - 0.309(156)logM - 2.374(+-1.092)(logM)^2